# Smart Agriculture Intelligence System — DMA Mini Project

**Google Colab notebook.** Run cells from top to bottom. The generated agricultural and weather datasets are synthetic demonstration data, not real farm measurements.


In [ ]:
!pip -q install numpy pandas matplotlib seaborn scikit-learn statsmodels joblib ipywidgets


## ============================================================


In [ ]:
# ============================================================
# SMART AGRICULTURE INTELLIGENCE SYSTEM
# DMA Mini Project | Google Colab-ready Python script
# Modules: EDA, preprocessing, PCA, LDA, Random Forest,
#          K-Means clustering, ARIMA forecasting, dashboard
# NOTE: This version generates SYNTHETIC demonstration data.
# Replace it with validated real farm/weather data for real use.
# ============================================================


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from statsmodels.tsa.arima.model import ARIMA

from IPython.display import display, clear_output
import ipywidgets as widgets

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 70)
print("             SMART AGRICULTURE INTELLIGENCE SYSTEM")
print("=" * 70)


## 1. DATASET GENERATION (SYNTHETIC)


In [ ]:
# 1. DATASET GENERATION (SYNTHETIC)


## print("\n[1] Generating synthetic agricultural dataset...")


In [ ]:
print("\n[1] Generating synthetic agricultural dataset...")

n_samples = 2000

crop_profiles = {
    "Rice":       [90, 45, 45, 28, 80, 6.2, 220],
    "Wheat":      [80, 40, 40, 20, 55, 6.8, 90],
    "Maize":      [85, 50, 45, 26, 65, 6.5, 130],
    "Cotton":     [70, 40, 55, 30, 55, 7.0, 100],
    "Sugarcane":  [110, 60, 70, 29, 75, 6.8, 200],
    "Groundnut":  [35, 55, 40, 27, 60, 6.3, 100],
    "Soybean":    [45, 60, 45, 25, 65, 6.5, 130],
    "Millets":    [35, 25, 30, 32, 40, 7.0, 65],
    "Pulses":     [30, 45, 35, 25, 55, 6.7, 85],
    "Potato":     [90, 55, 85, 19, 70, 5.8, 110]
}

feature_cols = [
    "N", "P", "K", "temperature",
    "humidity", "ph", "rainfall"
]

yield_base = {
    "Rice": 4.0, "Wheat": 3.5, "Maize": 5.0,
    "Cotton": 2.0, "Sugarcane": 70.0,
    "Groundnut": 2.5, "Soybean": 2.8,
    "Millets": 2.0, "Pulses": 1.8, "Potato": 25.0
}

records = []
noise_sd = np.array([12, 10, 10, 2.5, 7, 0.35, 20])

for _ in range(n_samples):
    crop_name = np.random.choice(list(crop_profiles.keys()))
    vals = np.array(crop_profiles[crop_name], dtype=float)
    vals += np.random.normal(0, noise_sd)
    vals = np.clip(
        vals,
        [0, 0, 0, 5, 10, 3.5, 0],
        [150, 150, 150, 45, 100, 9, 350]
    )
    row = dict(zip(feature_cols, vals))
    row["crop"] = crop_name
    records.append(row)

data = pd.DataFrame(records)

# Synthetic yield target; it is not measured farm yield.
def make_yield(row):
    base = yield_base[row["crop"]]
    value = base * (
        1
        + 0.001 * (row["N"] - 70)
        + 0.0005 * (row["rainfall"] - 120)
        - 0.015 * abs(row["ph"] - 6.5)
    )
    value += np.random.normal(0, base * 0.06)
    return max(0.1, value)

data["yield"] = data.apply(make_yield, axis=1)
data.to_csv("agriculture_dataset.csv", index=False)

print("Dataset shape:", data.shape)
print("Dataset source: synthetically generated for demonstration.")
display(data.head())


## 2. EXPLORATORY DATA ANALYSIS (EDA)


In [ ]:
# 2. EXPLORATORY DATA ANALYSIS (EDA)


## print("\n[2] Exploratory Data Analysis")


In [ ]:
print("\n[2] Exploratory Data Analysis")
print("\nData types:")
print(data.dtypes)
print("\nMissing values:")
print(data.isnull().sum())
print("\nDescriptive statistics:")
display(data.describe().round(2))

plt.figure(figsize=(10, 6))
sns.heatmap(data[feature_cols + ["yield"]].corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Agricultural Features")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.countplot(data=data, x="crop", order=data["crop"].value_counts().index)
plt.xticks(rotation=35)
plt.title("Crop Class Distribution")
plt.tight_layout()
plt.show()


## 3. PREPROCESSING


In [ ]:
# 3. PREPROCESSING


## print("\n[3] Data preprocessing")


In [ ]:
print("\n[3] Data preprocessing")

# The generated dataset has no intentionally inserted missing values.
# These steps are included so the pipeline can handle missing values
# if a real CSV is substituted.
data[feature_cols] = data[feature_cols].apply(
    lambda col: col.fillna(col.median())
)
data["crop"] = data["crop"].fillna(data["crop"].mode()[0])

# One-hot encode crop for the yield model.
yield_X = pd.get_dummies(data[feature_cols + ["crop"]],
                         columns=["crop"], dtype=int)
yield_feature_names = yield_X.columns.tolist()
yield_y = data["yield"]


## 4. CROP YIELD PREDICTION - RANDOM FOREST REGRESSION


In [ ]:
# 4. CROP YIELD PREDICTION - RANDOM FOREST REGRESSION


## print("\n[4] Crop yield prediction - Random Forest Regression")


In [ ]:
print("\n[4] Crop yield prediction - Random Forest Regression")

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(
    yield_X, yield_y, test_size=0.2, random_state=RANDOM_STATE
)

yield_model = RandomForestRegressor(
    n_estimators=150, max_depth=15,
    random_state=RANDOM_STATE, n_jobs=-1
)
yield_model.fit(X_train_y, y_train_y)
yield_pred = yield_model.predict(X_test_y)

yield_mae = mean_absolute_error(y_test_y, yield_pred)
yield_rmse = np.sqrt(mean_squared_error(y_test_y, yield_pred))
yield_r2 = r2_score(y_test_y, yield_pred)

print(f"MAE : {yield_mae:.3f}")
print(f"RMSE: {yield_rmse:.3f}")
print(f"R2  : {yield_r2:.3f}")

plt.figure(figsize=(6, 5))
plt.scatter(y_test_y, yield_pred, alpha=0.5)
lo = min(y_test_y.min(), yield_pred.min())
hi = max(y_test_y.max(), yield_pred.max())
plt.plot([lo, hi], [lo, hi], "r--")
plt.xlabel("Synthetic actual yield")
plt.ylabel("Predicted yield")
plt.title("Actual vs Predicted Yield")
plt.tight_layout()
plt.show()


## 5. CROP RECOMMENDATION - RANDOM FOREST CLASSIFICATION


In [ ]:
# 5. CROP RECOMMENDATION - RANDOM FOREST CLASSIFICATION


## print("\n[5] Crop recommendation - Random Forest Classification")


In [ ]:
print("\n[5] Crop recommendation - Random Forest Classification")

X = data[feature_cols]
y = data["crop"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Fit scaler only on training data to avoid data leakage.
classification_scaler = StandardScaler()
X_train_scaled = classification_scaler.fit_transform(X_train)
X_test_scaled = classification_scaler.transform(X_test)

crop_model = RandomForestClassifier(
    n_estimators=200, max_depth=15,
    random_state=RANDOM_STATE, n_jobs=-1
)
crop_model.fit(X_train, y_train)
crop_pred = crop_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, crop_pred)
print(f"Random Forest accuracy: {baseline_accuracy * 100:.2f}%")
print(classification_report(y_test, crop_pred, zero_division=0))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, crop_pred,
                             labels=crop_model.classes_),
            annot=True, fmt="d", cmap="Blues",
            xticklabels=crop_model.classes_,
            yticklabels=crop_model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()


## 6. PCA - UNSUPERVISED DIMENSIONALITY REDUCTION


In [ ]:
# 6. PCA - UNSUPERVISED DIMENSIONALITY REDUCTION


## print("\n[6] PCA dimensionality reduction")


In [ ]:
print("\n[6] PCA dimensionality reduction")

pca_scaler = StandardScaler()
X_train_pca_scaled = pca_scaler.fit_transform(X_train)
X_test_pca_scaled = pca_scaler.transform(X_test)

# Retain enough components to explain at least 95% variance.
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_pca_scaled)
X_test_pca = pca.transform(X_test_pca_scaled)

print("Original feature count:", X_train.shape[1])
print("PCA component count:", X_train_pca.shape[1])
print("Total explained variance:",
      round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

pca_rf = RandomForestClassifier(
    n_estimators=200, max_depth=15,
    random_state=RANDOM_STATE, n_jobs=-1
)
pca_rf.fit(X_train_pca, y_train)
pca_pred = pca_rf.predict(X_test_pca)
pca_accuracy = accuracy_score(y_test, pca_pred)
print(f"Random Forest after PCA accuracy: {pca_accuracy * 100:.2f}%")

plt.figure(figsize=(7, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.axhline(0.95, color="red", linestyle="--", label="95% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Two-component PCA plot for visualization only.
pca_2d = PCA(n_components=2)
X_all_scaled_for_plot = StandardScaler().fit_transform(X)
X_pca_2d = pca_2d.fit_transform(X_all_scaled_for_plot)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                      c=pd.factorize(y)[0], alpha=0.6, cmap="tab10")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Agricultural Dataset Visualized using PCA")
plt.tight_layout()
plt.show()


## 7. LDA - SUPERVISED DIMENSIONALITY REDUCTION


In [ ]:
# 7. LDA - SUPERVISED DIMENSIONALITY REDUCTION


## print("\n[7] LDA dimensionality reduction")


In [ ]:
print("\n[7] LDA dimensionality reduction")

# For C classes, LDA can produce at most C-1 components.
n_classes = y_train.nunique()
lda_components = min(n_classes - 1, X_train.shape[1])

lda_scaler = StandardScaler()
X_train_lda_scaled = lda_scaler.fit_transform(X_train)
X_test_lda_scaled = lda_scaler.transform(X_test)

lda = LinearDiscriminantAnalysis(n_components=lda_components)
X_train_lda = lda.fit_transform(X_train_lda_scaled, y_train)
X_test_lda = lda.transform(X_test_lda_scaled)

lda_classifier = LogisticRegression(max_iter=2000)
lda_classifier.fit(X_train_lda, y_train)
lda_pred = lda_classifier.predict(X_test_lda)
lda_accuracy = accuracy_score(y_test, lda_pred)

print("Number of LDA components:", lda_components)
print(f"LDA + Logistic Regression accuracy: {lda_accuracy * 100:.2f}%")
print("\nComparison:")
print(f"Original features + Random Forest: {baseline_accuracy * 100:.2f}%")
print(f"PCA features + Random Forest     : {pca_accuracy * 100:.2f}%")
print(f"LDA features + Logistic Regression: {lda_accuracy * 100:.2f}%")

if X_train_lda.shape[1] >= 2:
    plt.figure(figsize=(8, 5))
    plt.scatter(X_train_lda[:, 0], X_train_lda[:, 1],
                c=pd.factorize(y_train)[0], alpha=0.6, cmap="tab10")
    plt.xlabel("LD1")
    plt.ylabel("LD2")
    plt.title("Crop Classes Visualized using LDA")
    plt.tight_layout()
    plt.show()


## 8. AGRICULTURAL PATTERN DETECTION - K-MEANS


In [ ]:
# 8. AGRICULTURAL PATTERN DETECTION - K-MEANS


## print("\n[8] Agricultural pattern detection - K-Means")


In [ ]:
print("\n[8] Agricultural pattern detection - K-Means")

cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(data[feature_cols])

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
data["cluster"] = kmeans.fit_predict(X_cluster_scaled)

sil_score = silhouette_score(
    X_cluster_scaled, data["cluster"],
    sample_size=min(1000, len(data)), random_state=RANDOM_STATE
)
print("Silhouette score:", round(sil_score, 3))
print("\nCluster profiles:")
display(data.groupby("cluster")[feature_cols + ["yield"]].mean().round(2))

plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="rainfall", y="temperature",
                hue="cluster", palette="viridis", alpha=0.7)
plt.title("Agricultural Environmental Clusters")
plt.tight_layout()
plt.show()


## 9. WEATHER FORECASTING - ARIMA


In [ ]:
# 9. WEATHER FORECASTING - ARIMA


## print("\n[9] Weather forecasting - ARIMA")


In [ ]:
print("\n[9] Weather forecasting - ARIMA")

# Synthetic daily weather series for demonstration only.
dates = pd.date_range("2023-01-01", periods=1095, freq="D")
day = np.arange(len(dates))

weather = pd.DataFrame({
    "temperature": 27 + 6*np.sin(2*np.pi*day/365)
                   + np.random.normal(0, 2, len(day)),
    "rainfall": np.maximum(
        0, 4 + 5*np.sin(2*np.pi*(day-170)/365)
        + np.random.normal(0, 3, len(day))
    ),
    "humidity": np.clip(
        65 + 15*np.sin(2*np.pi*(day-170)/365)
        + np.random.normal(0, 5, len(day)), 20, 100
    )
}, index=dates)
weather.index.name = "date"

forecast_horizon = 30
forecast_results = {}
forecast_mae = {}

for col in ["temperature", "rainfall", "humidity"]:
    train_series = weather[col].iloc[:-forecast_horizon]
    test_series = weather[col].iloc[-forecast_horizon:]

    eval_fit = ARIMA(train_series, order=(2, 1, 2)).fit()
    test_forecast = eval_fit.forecast(steps=forecast_horizon)
    forecast_mae[col] = mean_absolute_error(test_series, test_forecast)

    final_fit = ARIMA(weather[col], order=(2, 1, 2)).fit()
    forecast_results[col] = final_fit.forecast(
        steps=forecast_horizon
    ).to_numpy()

    print(f"{col} test MAE: {forecast_mae[col]:.3f}")

future_dates = pd.date_range(
    weather.index[-1] + pd.Timedelta(days=1),
    periods=forecast_horizon, freq="D"
)
forecast_df = pd.DataFrame(forecast_results, index=future_dates)
forecast_df.index.name = "date"
forecast_df["rainfall"] = forecast_df["rainfall"].clip(lower=0)
forecast_df["humidity"] = forecast_df["humidity"].clip(0, 100)

display(forecast_df.head(10).round(2))

for col in ["temperature", "rainfall", "humidity"]:
    plt.figure(figsize=(10, 4))
    plt.plot(weather.index[-90:], weather[col].iloc[-90:],
             label="Synthetic history")
    plt.plot(forecast_df.index, forecast_df[col],
             "--", label="ARIMA forecast")
    plt.title(f"{col.title()} - 30 Day Forecast")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## 10. INTERACTIVE AGRICULTURE PREDICTION


In [ ]:
# 10. INTERACTIVE AGRICULTURE PREDICTION


## print("\n[10] Interactive agriculture dashboard")


In [ ]:
print("\n[10] Interactive agriculture dashboard")

def analyze_farm(N, P, K, temperature, humidity, ph, rainfall, selected_crop):
    sample = pd.DataFrame([{
        "N": N, "P": P, "K": K,
        "temperature": temperature,
        "humidity": humidity, "ph": ph,
        "rainfall": rainfall
    }])

    recommended_crop = crop_model.predict(sample)[0]
    probabilities = crop_model.predict_proba(sample)[0]
    top_idx = np.argsort(probabilities)[::-1][:3]

    yield_row = sample.copy()
    yield_row["crop"] = selected_crop
    yield_row = pd.get_dummies(yield_row, columns=["crop"], dtype=int)
    yield_row = yield_row.reindex(columns=yield_feature_names, fill_value=0)
    predicted_yield = yield_model.predict(yield_row)[0]

    cluster_id = int(kmeans.predict(
        cluster_scaler.transform(sample[feature_cols])
    )[0])

    print("=" * 55)
    print("SMART AGRICULTURE INTELLIGENCE REPORT")
    print("=" * 55)
    print(f"Selected crop for yield estimate: {selected_crop}")
    print(f"Estimated synthetic yield: {predicted_yield:.2f} (dataset units)")
    print(f"Recommended crop: {recommended_crop}")
    print("\nTop 3 model recommendations:")
    for idx in top_idx:
        print(f"  {crop_model.classes_[idx]}: {probabilities[idx]*100:.2f}%")
    print(f"\nAgricultural cluster: {cluster_id}")
    print("Cluster ID represents a group of similar input conditions.")
    print("\nSynthetic 30-day forecast averages:")
    print(f"Temperature: {forecast_df['temperature'].mean():.2f} °C")
    print(f"Rainfall total: {forecast_df['rainfall'].sum():.2f} mm")
    print(f"Humidity: {forecast_df['humidity'].mean():.2f}%")
    print("\nNOTE: Demonstration output based on synthetic data.")
    print("Do not use these estimates for actual farm decisions.")

style = {"description_width": "130px"}
wN = widgets.IntSlider(value=80, min=0, max=150, description="Nitrogen", style=style)
wP = widgets.IntSlider(value=40, min=0, max=150, description="Phosphorus", style=style)
wK = widgets.IntSlider(value=40, min=0, max=150, description="Potassium", style=style)
wT = widgets.FloatSlider(value=28, min=5, max=45, description="Temperature", style=style)
wH = widgets.FloatSlider(value=75, min=10, max=100, description="Humidity", style=style)
wPH = widgets.FloatSlider(value=6.5, min=3.5, max=9, step=0.1, description="Soil pH", style=style)
wR = widgets.FloatSlider(value=180, min=0, max=350, description="Rainfall", style=style)
wCrop = widgets.Dropdown(options=list(crop_profiles.keys()),
                         value="Rice", description="Selected crop", style=style)
button = widgets.Button(description="Analyze Agriculture Data",
                        button_style="success", icon="check")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output(wait=True)
        analyze_farm(wN.value, wP.value, wK.value,
                     wT.value, wH.value, wPH.value,
                     wR.value, wCrop.value)

button.on_click(on_click)

display(widgets.HTML("<h2>Smart Agriculture Intelligence Dashboard</h2>"))
display(wN, wP, wK, wT, wH, wPH, wR, wCrop, button, out)


## 11. SAVE MODELS, TRANSFORMERS AND OUTPUT DATA


In [ ]:
# 11. SAVE MODELS, TRANSFORMERS AND OUTPUT DATA


## print("\n[11] Saving project artifacts")


In [ ]:
print("\n[11] Saving project artifacts")

joblib.dump(yield_model, "crop_yield_model.pkl")
joblib.dump(yield_feature_names, "yield_feature_columns.pkl")
joblib.dump(crop_model, "crop_recommendation_model.pkl")
joblib.dump(classification_scaler, "classification_scaler.pkl")
joblib.dump(pca, "pca_model.pkl")
joblib.dump(pca_scaler, "pca_scaler.pkl")
joblib.dump(lda, "lda_model.pkl")
joblib.dump(lda_scaler, "lda_scaler.pkl")
joblib.dump(kmeans, "agriculture_kmeans_model.pkl")
joblib.dump(cluster_scaler, "agriculture_cluster_scaler.pkl")

data.to_csv("agriculture_dataset_with_clusters.csv", index=False)
weather.to_csv("synthetic_historical_weather.csv")
forecast_df.to_csv("synthetic_weather_forecast.csv")

print("Saved models, scalers, dataset and forecast files.")
print("Project execution completed.")
print("\nIMPORTANT: All generated data in this script is synthetic.")
